# 📊 Google Colab + Kaggle + SQL
## Opérations CRUD Complètes avec Données Réelles

---

### 🎯 Objectifs de ce Notebook
- Télécharger un dataset Kaggle
- Décrire la structure de la base de données
- Tester des requêtes INSERT
- Tester des requêtes SELECT
- Tester des requêtes UPDATE
- Tester des requêtes DELETE

**Temps estimé:** 15-20 minutes

---

## Section 1: Installation des Dépendances

Installez automatiquement tous les packages nécessaires.

In [31]:
# Installation des dépendances
import subprocess
import sys
import os

# Désactiver temporairement l'authentification automatique lors de l'import
os.environ['KAGGLE_CONFIG_DIR'] = os.path.expanduser('~/.kaggle')

packages = ['kaggle', 'pandas', 'numpy', 'matplotlib', 'seaborn']

print("Installation des dépendances...\n")

for package in packages:
    try:
        if package == 'kaggle':
            # Pour kaggle, on vérifie via pip ou import sans déclencher l'authentification immédiate
            import importlib.util
            spec = importlib.util.find_spec(package)
            if spec is None:
                raise ImportError
        else:
            __import__(package)
        print(f"✓ {package} est déjà installé")
    except (ImportError, NameError):
        print(f"Installation de {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installé")

print("\n✅ Toutes les dépendances sont prêtes!")

Installation des dépendances...

✓ kaggle est déjà installé
✓ pandas est déjà installé
✓ numpy est déjà installé
✓ matplotlib est déjà installé
✓ seaborn est déjà installé

✅ Toutes les dépendances sont prêtes!


## Section 2: Configuration Kaggle

Authentifiez-vous avec votre clé API Kaggle.

In [32]:
import os

# Définir le token API
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_d7c448a1be6881316a06187f98dcf90a'

# Puis utiliser Kaggle comme normal
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

print("✓ Authentification réussie!")

✓ Authentification réussie!


In [34]:
import os
import json

# Créer le dossier .kaggle
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# Créer le fichier avec votre token
token_data = {
    "username": "votreusername",  # À remplir avec votre username Kaggle
    "key": "KGAT_d7c448a1be6881316a06187f98dcf90a"
}

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(token_data, f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print("✓ kaggle.json créé!")

✓ kaggle.json créé!


In [35]:
import os
from google.colab import files

# Configuration du répertoire Kaggle
kaggle_dir = os.path.expanduser('~/.kaggle')
if not os.path.exists(kaggle_dir):
    os.makedirs(kaggle_dir)
    print(f"✓ Dossier {kaggle_dir} créé")

# Vérifier la présence de kaggle.json
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
if not os.path.exists(kaggle_json_path):
    print("\n⚠️  Fichier kaggle.json non trouvé!")
    print("\nInstructions:")
    print("1. Allez sur https://www.kaggle.com/account")
    print("2. Cliquez 'Create New API Token'")
    print("3. Téléchargez kaggle.json")
    print("4. Uploadez le fichier ci-dessous:\n")

    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        os.rename('kaggle.json', kaggle_json_path)
        os.chmod(kaggle_json_path, 0o600)
        print("\n✓ kaggle.json configuré avec succès")
else:
    print("✓ Fichier kaggle.json trouvé")

# Test d'authentification
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
print("✓ Authentification Kaggle réussie")

✓ Fichier kaggle.json trouvé
✓ Authentification Kaggle réussie


## Section 3: Connexion Google Drive

Montez Google Drive pour sauvegarder les données.

In [36]:
import os
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive monté")

# Créer le dossier pour nos données
drive_data_dir = '/content/drive/MyDrive/Colab Datasets'
if not os.path.exists(drive_data_dir):
    os.makedirs(drive_data_dir)
    print(f"✓ Dossier créé: {drive_data_dir}")
else:
    print(f"✓ Dossier existe déjà: {drive_data_dir}")

Mounted at /content/drive
✓ Google Drive monté
✓ Dossier existe déjà: /content/drive/MyDrive/Colab Datasets


In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 4: Téléchargement du Dataset Kaggle

Téléchargez le dataset Titanic depuis Kaggle.

In [39]:
# Téléchargement du dataset Titanic via une URL publique
import urllib.request
import os

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
destination_file = os.path.join(drive_data_dir, 'train.csv')

print(f"Téléchargement du dataset depuis l'URL publique...\n")

try:
    # Télécharger le fichier et le sauvegarder sous le nom 'train.csv'
    urllib.request.urlretrieve(url, destination_file)
    print(f"✓ Dataset téléchargé avec succès et enregistré sous : {destination_file}\n")

    # Lister les fichiers pour vérification
    print("Fichiers dans le dossier :")
    files_list = os.listdir(drive_data_dir)
    for file in files_list:
        file_path = os.path.join(drive_data_dir, file)
        if os.path.isfile(file_path):
            file_size = os.path.getsize(file_path)
            print(f"  - {file} ({file_size:,} bytes)")
except Exception as e:
    print(f"Une erreur est survenue lors du téléchargement : {str(e)}")

Téléchargement du dataset depuis l'URL publique...

✓ Dataset téléchargé avec succès et enregistré sous : /content/drive/MyDrive/Colab Datasets/train.csv

Fichiers dans le dossier :
  - train.csv (60,302 bytes)
  - titanic_database.db (73,728 bytes)


## Section 5: Création de la Base de Données SQLite

In [40]:
import sqlite3
import pandas as pd

# Chemins des fichiers
db_path = os.path.join(drive_data_dir, 'titanic_database.db')
train_csv = os.path.join(drive_data_dir, 'train.csv')

# Supprimer l'ancienne base si elle existe
if os.path.exists(db_path):
    os.remove(db_path)
    print("Ancienne base de données supprimée")

# Créer la connexion
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"✓ Base de données créée: {db_path}\n")

# Charger les données
df = pd.read_csv(train_csv)
print(f"✓ Données chargées: {len(df)} lignes, {len(df.columns)} colonnes\n")

# Créer la table
table_name = 'passengers'
df.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"✓ Table '{table_name}' créée dans la base de données")

Ancienne base de données supprimée
✓ Base de données créée: /content/drive/MyDrive/Colab Datasets/titanic_database.db

✓ Données chargées: 891 lignes, 12 colonnes

✓ Table 'passengers' créée dans la base de données


## Section 6: Description de la Structure de la Base de Données

In [42]:
# Informations sur la table
cursor.execute(f"PRAGMA table_info({table_name})")
columns_info = cursor.fetchall()

print("\n📋 STRUCTURE DE LA TABLE")
print("=" * 80)

columns_df = pd.DataFrame(columns_info, columns=['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk'])
print(columns_df[['name', 'type', 'notnull']].to_string(index=False))

# Statistiques
print("\n\n📊 STATISTIQUES")
print("=" * 80)

cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
total_rows = cursor.fetchone()[0]
print(f"Total des lignes: {total_rows:,}")

# Analyse détaillée
print("\n📈 ANALYSE PAR COLONNE")
print("=" * 80)

for col_info in columns_info[:8]:  # Afficher les 8 premières colonnes
    col_name = col_info[1]
    col_type = col_info[2]

    cursor.execute(f"SELECT COUNT(DISTINCT {col_name}) FROM {table_name} WHERE {col_name} IS NOT NULL")
    distinct_count = cursor.fetchone()[0]

    cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE {col_name} IS NULL")
    null_count = cursor.fetchone()[0]

    print(f"\n  {col_name} ({col_type}):")
    print(f"    Valeurs uniques: {distinct_count}")
    print(f"    Valeurs NULL: {null_count}")

    if col_type in ['INTEGER', 'REAL']:
        cursor.execute(f"SELECT MIN({col_name}), MAX({col_name}), AVG({col_name}) FROM {table_name}")
        min_val, max_val, avg_val = cursor.fetchone()
        if min_val is not None:
            print(f"    Min: {min_val}, Max: {max_val}, Moyenne: {avg_val:.2f}")


📋 STRUCTURE DE LA TABLE
       name    type  notnull
PassengerId INTEGER        0
   Survived INTEGER        0
     Pclass INTEGER        0
       Name    TEXT        0
        Sex    TEXT        0
        Age    REAL        0
      SibSp INTEGER        0
      Parch INTEGER        0
     Ticket    TEXT        0
       Fare    REAL        0
      Cabin    TEXT        0
   Embarked    TEXT        0


📊 STATISTIQUES
Total des lignes: 891

📈 ANALYSE PAR COLONNE

  PassengerId (INTEGER):
    Valeurs uniques: 891
    Valeurs NULL: 0
    Min: 1, Max: 891, Moyenne: 446.00

  Survived (INTEGER):
    Valeurs uniques: 2
    Valeurs NULL: 0
    Min: 0, Max: 1, Moyenne: 0.38

  Pclass (INTEGER):
    Valeurs uniques: 3
    Valeurs NULL: 0
    Min: 1, Max: 3, Moyenne: 2.31

  Name (TEXT):
    Valeurs uniques: 891
    Valeurs NULL: 0

  Sex (TEXT):
    Valeurs uniques: 2
    Valeurs NULL: 0

  Age (REAL):
    Valeurs uniques: 88
    Valeurs NULL: 177
    Min: 0.42, Max: 80.0, Moyenne: 29.70

  SibSp

## Aperçu des Données

In [43]:
print("\n📋 PREMIÈRES LIGNES DE LA TABLE")
print("=" * 80 + "\n")

query = f"SELECT * FROM {table_name} LIMIT 5"
result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 PREMIÈRES LIGNES DE LA TABLE



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


---

# 📝 OPÉRATIONS SQL

## Section 7: REQUÊTES INSERT

### 1. Insert Simple

In [44]:
print("\n1️⃣  INSERT SIMPLE")
print("=" * 80)

new_passenger = {
    'PassengerId': 892,
    'Survived': 1,
    'Pclass': 1,
    'Name': 'Claude, Monsieur',
    'Sex': 'male',
    'Age': 35.0,
    'SibSp': 1,
    'Parch': 1,
    'Ticket': 'AI001',
    'Fare': 512.33,
    'Cabin': 'A123',
    'Embarked': 'S'
}

insert_query = f"""
INSERT INTO {table_name}
(PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

values = tuple(new_passenger.values())
cursor.execute(insert_query, values)
conn.commit()

print("\n✓ Requête INSERT exécutée avec succès!")
print(f"\nDonnées insérées:")
for key, value in new_passenger.items():
    print(f"  {key}: {value}")

# Vérification
cursor.execute(f"SELECT * FROM {table_name} WHERE PassengerId = 892")
inserted_row = cursor.fetchone()
print(f"\n✓ Vérification: Ligne retrouvée dans la base de données")


1️⃣  INSERT SIMPLE

✓ Requête INSERT exécutée avec succès!

Données insérées:
  PassengerId: 892
  Survived: 1
  Pclass: 1
  Name: Claude, Monsieur
  Sex: male
  Age: 35.0
  SibSp: 1
  Parch: 1
  Ticket: AI001
  Fare: 512.33
  Cabin: A123
  Embarked: S

✓ Vérification: Ligne retrouvée dans la base de données


### 2. Insert Multiple (Bulk Insert)

In [45]:
print("\n2️⃣  INSERT MULTIPLE (BULK INSERT)")
print("=" * 80)

new_passengers = [
    (893, 0, 3, 'Dupont, Monsieur', 'male', 25.0, 0, 0, 'AI002', 7.75, None, 'C'),
    (894, 1, 1, 'Lefevre, Madame', 'female', 28.0, 1, 1, 'AI003', 151.86, 'B5', 'S'),
    (895, 0, 2, 'Martin, Monsieur', 'male', 45.0, 0, 0, 'AI004', 10.50, None, 'Q'),
]

bulk_insert_query = f"""
INSERT INTO {table_name}
(PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

cursor.executemany(bulk_insert_query, new_passengers)
conn.commit()

print("✓ Insertion en masse réussie!")
print(f"✓ {len(new_passengers)} passagers insérés\n")

print("Détail des insertions:")
for i, passenger in enumerate(new_passengers, 1):
    print(f"  Passager {i}: PassengerId={passenger[0]}, Name={passenger[3]}, Age={passenger[5]}")

# Vérifier
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
total_after = cursor.fetchone()[0]
print(f"\nTotal des lignes après insertion: {total_after:,}")


2️⃣  INSERT MULTIPLE (BULK INSERT)
✓ Insertion en masse réussie!
✓ 3 passagers insérés

Détail des insertions:
  Passager 1: PassengerId=893, Name=Dupont, Monsieur, Age=25.0
  Passager 2: PassengerId=894, Name=Lefevre, Madame, Age=28.0
  Passager 3: PassengerId=895, Name=Martin, Monsieur, Age=45.0

Total des lignes après insertion: 895


---

## Section 8: REQUÊTES SELECT

### 1. SELECT Simple

In [46]:
print("\n1️⃣  SELECT SIMPLE")
print("=" * 80 + "\n")

query = f"SELECT age FROM {table_name} WHERE PassengerId >= 890 LIMIT 5"
print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)


1️⃣  SELECT SIMPLE

Requête: SELECT * FROM passengers WHERE PassengerId >= 890 LIMIT 5



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
1,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,None,Q
2,892,1,1,"Claude, Monsieur",male,35.0,1,1,AI001,512.33,A123,S
3,893,0,3,"Dupont, Monsieur",male,25.0,0,0,AI002,7.75,None,C
4,894,1,1,"Lefevre, Madame",female,28.0,1,1,AI003,151.86,B5,S


### 2. SELECT avec Filtres

In [47]:
print("\n2️⃣  SELECT AVEC FILTRES")
print("=" * 80 + "\n")

query = f"""
SELECT Name, Age, Pclass, Fare
FROM {table_name}
WHERE Age > 30 AND Survived = 1
ORDER BY Age DESC
LIMIT 10
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
print(f"Résultat: {len(result_df)} passagers trouvés\n")
display(result_df)


2️⃣  SELECT AVEC FILTRES

Requête: 
SELECT Name, Age, Pclass, Fare
FROM passengers
WHERE Age > 30 AND Survived = 1
ORDER BY Age DESC
LIMIT 10


Résultat: 10 passagers trouvés



,Name,Age,Pclass,Fare
0,"Barkworth, Mr. Algernon Henry Wilson",80.0,1,30.0000
1,"Andrews, Miss. Kornelia Theodosia",63.0,1,77.9583
2,"Turkula, Mrs. (Hedwig)",63.0,3,9.5875
3,"Harris, Mr. George",62.0,2,10.5000
4,"Stone, Mrs. George Nelson (Martha Evelyn)",62.0,1,80.0000
5,"Warren, Mrs. Frank Manley (Anna Sophia Atkinson)",60.0,1,75.2500
6,"Frolicher-Stehli, Mr. Maxmillian",60.0,1,79.2000
7,"Bonnell, Miss. Elizabeth",58.0,1,26.5500
8,"Lurette, Miss. Elise",58.0,1,146.5208
9,"Graham, Mrs. William Thompson (Edith Junkins)",58.0,1,153.4625


### 3. SELECT avec Agrégation

In [48]:
print("\n3️⃣  SELECT AVEC AGRÉGATION")
print("=" * 80 + "\n")

query = f"""
SELECT
    Survived,
    COUNT(*) as nb_passagers,
    ROUND(AVG(Age), 2) as age_moyen,
    ROUND(AVG(Fare), 2) as fare_moyen
FROM {table_name}
GROUP BY Survived
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)


3️⃣  SELECT AVEC AGRÉGATION

Requête: 
SELECT
    Survived,
    COUNT(*) as nb_passagers,
    ROUND(AVG(Age), 2) as age_moyen,
    ROUND(AVG(Fare), 2) as fare_moyen
FROM passengers
GROUP BY Survived




,Survived,nb_passagers,age_moyen,fare_moyen
0,0,551,30.65,22.07
1,1,344,28.37,50.04


### 4. SELECT Statistiques par Classe

In [ ]:
print("\n4️⃣  STATISTIQUES PAR CLASSE")
print("=" * 80 + "\n")

query = f"""
SELECT
    Pclass as classe,
    COUNT(*) as nb_passagers,
    SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) as survivants,
    ROUND(100.0 * SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as taux_survie_pct
FROM {table_name}
WHERE Pclass IS NOT NULL
GROUP BY Pclass
ORDER BY Pclass
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)


4️⃣  STATISTIQUES PAR CLASSE

Requête: 
SELECT
    Pclass as classe,
    COUNT(*) as nb_passagers,
    SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) as survivants,
    ROUND(100.0 * SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as taux_survie_pct
FROM passengers
WHERE Pclass IS NOT NULL
GROUP BY Pclass
ORDER BY Pclass




,classe,nb_passagers,survivants,taux_survie_pct
0,1,218,138,63.30
1,2,185,87,47.03
2,3,492,119,24.19


### 5. SELECT Top 10 (Les plus riches)

In [ ]:
print("\n5️⃣  TOP 10 PASSAGERS LES PLUS RICHES")
print("=" * 80 + "\n")

query = f"""
SELECT
    Name,
    Age,
    Sex,
    Fare,
    Survived
FROM {table_name}
WHERE Fare IS NOT NULL
ORDER BY Fare DESC
LIMIT 10
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)


5️⃣  TOP 10 PASSAGERS LES PLUS RICHES

Requête: 
SELECT
    Name,
    Age,
    Sex,
    Fare,
    Survived
FROM passengers
WHERE Fare IS NOT NULL
ORDER BY Fare DESC
LIMIT 10




,Name,Age,Sex,Fare,Survived
0,"Claude, Monsieur",35.0,male,512.3300,1
1,"Ward, Miss. Anna",35.0,female,512.3292,1
2,"Cardeza, Mr. Thomas Drake Martinez",36.0,male,512.3292,1
3,"Lesurer, Mr. Gustave J",35.0,male,512.3292,1
4,"Fortune, Mr. Charles Alexander",19.0,male,263.0000,0
5,"Fortune, Miss. Mabel Helen",23.0,female,263.0000,1
6,"Fortune, Miss. Alice Elizabeth",24.0,female,263.0000,1
7,"Fortune, Mr. Mark",64.0,male,263.0000,0
8,"Ryerson, Miss. Emily Borie",18.0,female,262.3750,1
9,"Ryerson, Miss. Susan Parker ""Suzette""",21.0,female,262.3750,1


---

## Section 9: REQUÊTES UPDATE

### 1. Update Simple

In [49]:
print("\n1️⃣  UPDATE SIMPLE")
print("=" * 80 + "\n")

# Avant
cursor.execute(f"SELECT PassengerId, Name, Age, Fare FROM {table_name} WHERE PassengerId = 892")
before = cursor.fetchone()
print("Avant modification:")
print(f"  PassengerId: {before[0]}, Name: {before[1]}, Age: {before[2]}, Fare: {before[3]}")

# Update
update_query = f"""
UPDATE {table_name}
SET Age = 40, Fare = 550.00
WHERE PassengerId = 892
"""

print(f"\nRequête UPDATE: {update_query}")
cursor.execute(update_query)
conn.commit()
print("✓ Modification réussie!")

# Après
cursor.execute(f"SELECT PassengerId, Name, Age, Fare FROM {table_name} WHERE PassengerId = 892")
after = cursor.fetchone()
print("\nAprès modification:")
print(f"  PassengerId: {after[0]}, Name: {after[1]}, Age: {after[2]}, Fare: {after[3]}")


1️⃣  UPDATE SIMPLE

Avant modification:
  PassengerId: 892, Name: Claude, Monsieur, Age: 35.0, Fare: 512.33

Requête UPDATE: 
UPDATE passengers
SET Age = 40, Fare = 550.00
WHERE PassengerId = 892

✓ Modification réussie!

Après modification:
  PassengerId: 892, Name: Claude, Monsieur, Age: 40.0, Fare: 550.0


### 2. Update Multiple Lignes

In [50]:
print("\n2️⃣  UPDATE PLUSIEURS LIGNES")
print("=" * 80 + "\n")

# Avant
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Age IS NULL AND Pclass = 1")
before_count = cursor.fetchone()[0]

update_query = f"""
UPDATE {table_name}
SET Age = 30
WHERE Age IS NULL AND Pclass = 1
"""

print(f"Requête UPDATE: {update_query}")
print(f"Lignes affectées avant: {before_count}")

cursor.execute(update_query)
conn.commit()
rows_affected = cursor.rowcount
print(f"✓ {rows_affected} lignes mises à jour")

# Après
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Age = 30 AND Pclass = 1")
after_count = cursor.fetchone()[0]
print(f"Lignes avec Age=30 et Pclass=1: {after_count}")


2️⃣  UPDATE PLUSIEURS LIGNES

Requête UPDATE: 
UPDATE passengers
SET Age = 30
WHERE Age IS NULL AND Pclass = 1

Lignes affectées avant: 30
✓ 30 lignes mises à jour
Lignes avec Age=30 et Pclass=1: 36


### 3. Update Conditionnel Complexe

In [51]:
print("\n3️⃣  UPDATE CONDITIONNEL COMPLEXE")
print("=" * 80 + "\n")

update_query = f"""
UPDATE {table_name}
SET Fare = Fare * 1.1
WHERE Pclass = 1 AND Fare IS NOT NULL
"""

print(f"Requête UPDATE: {update_query}")
print("(Augmentation de 10% des tarifs pour la première classe)\n")

cursor.execute(update_query)
conn.commit()
print(f"✓ {cursor.rowcount} lignes mises à jour")

# Aperçu
query = f"""
SELECT Name, Pclass, Fare FROM {table_name}
WHERE Pclass = 1 AND Fare IS NOT NULL
LIMIT 5
"""
result_df = pd.read_sql_query(query, conn)
print("\nAperçu des premières classes mises à jour:")
display(result_df)


3️⃣  UPDATE CONDITIONNEL COMPLEXE

Requête UPDATE: 
UPDATE passengers
SET Fare = Fare * 1.1
WHERE Pclass = 1 AND Fare IS NOT NULL

(Augmentation de 10% des tarifs pour la première classe)

✓ 218 lignes mises à jour

Aperçu des premières classes mises à jour:


,Name,Pclass,Fare
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,78.41163
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,58.41000
2,"McCarthy, Mr. Timothy J",1,57.04875
3,"Bonnell, Miss. Elizabeth",1,29.20500
4,"Sloper, Mr. William Thompson",1,39.05000


---

## Section 10: REQUÊTES DELETE

### 1. Delete Simple

In [ ]:
print("\n1️⃣  DELETE SIMPLE")
print("=" * 80 + "\n")

# Avant suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE PassengerId = 893")
exists_before = cursor.fetchone()[0]
print(f"Ligne avec PassengerId=893 existe avant suppression: {bool(exists_before)}")

# Supprimer
delete_query = f"DELETE FROM {table_name} WHERE PassengerId = 893"
print(f"\nRequête DELETE: {delete_query}")

cursor.execute(delete_query)
conn.commit()
print(f"✓ {cursor.rowcount} ligne supprimée")

# Après suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE PassengerId = 893")
exists_after = cursor.fetchone()[0]
print(f"Ligne avec PassengerId=893 existe après suppression: {bool(exists_after)}")


1️⃣  DELETE SIMPLE

Ligne avec PassengerId=893 existe avant suppression: True

Requête DELETE: DELETE FROM passengers WHERE PassengerId = 893
✓ 1 ligne supprimée
Ligne avec PassengerId=893 existe après suppression: False


### 2. Delete Multiple Lignes

In [ ]:
print("\n2️⃣  DELETE PLUSIEURS LIGNES")
print("=" * 80 + "\n")

# Insérer des lignes de test
test_passengers = [
    (896, 0, 3, 'Test1, Monsieur', 'male', 20.0, 0, 0, 'TEST1', 5.00, None, 'C'),
    (897, 1, 3, 'Test2, Madame', 'female', 22.0, 0, 0, 'TEST2', 7.75, None, 'S'),
    (898, 0, 3, 'Test3, Monsieur', 'male', 18.0, 0, 0, 'TEST3', 6.50, None, 'Q'),
]

cursor.executemany(bulk_insert_query, test_passengers)
conn.commit()
print("✓ 3 lignes de test insérées")

# Avant suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Ticket LIKE 'TEST%'")
before_delete = cursor.fetchone()[0]
print(f"Lignes de test avant suppression: {before_delete}")

# Supprimer
delete_query = f"DELETE FROM {table_name} WHERE Ticket LIKE 'TEST%'"
print(f"\nRequête DELETE: {delete_query}")

cursor.execute(delete_query)
conn.commit()
print(f"✓ {cursor.rowcount} lignes supprimées")

# Après suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Ticket LIKE 'TEST%'")
after_delete = cursor.fetchone()[0]
print(f"Lignes de test après suppression: {after_delete}")


2️⃣  DELETE PLUSIEURS LIGNES

✓ 3 lignes de test insérées
Lignes de test avant suppression: 3

Requête DELETE: DELETE FROM passengers WHERE Ticket LIKE 'TEST%'
✓ 3 lignes supprimées
Lignes de test après suppression: 0


---

## Section 11: Résumé et Statistiques Finales

In [ ]:
print("\n" + "=" * 80)
print("RÉSUMÉ ET STATISTIQUES FINALES")
print("=" * 80)

cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
final_count = cursor.fetchone()[0]

print(f"\n📊 RÉSUMÉ DES OPÉRATIONS:")
print("-" * 80)
print(f"✓ Base de données créée: {db_path}")
print(f"✓ Dataset téléchargé: {dataset_name}")
print(f"✓ Table créée: {table_name}")
print(f"✓ Total des lignes finales: {final_count:,}")
print(f"✓ Total des colonnes: {len(columns_info)}")

print(f"\n📈 OPÉRATIONS EFFECTUÉES:")
print("-" * 80)
print(f"  • INSERT: 4 passagers insérés")
print(f"  • SELECT: 5 requêtes de sélection exécutées")
print(f"  • UPDATE: 3 requêtes de mise à jour exécutées")
print(f"  • DELETE: 2 requêtes de suppression exécutées")

print(f"\n📋 STATISTIQUES FINALES:")
print("-" * 80)

query = f"""
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) as survivants,
    ROUND(100.0 * SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as taux_survie,
    ROUND(AVG(Age), 2) as age_moyen,
    ROUND(AVG(Fare), 2) as fare_moyen
FROM {table_name}
WHERE Age IS NOT NULL
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)

# Fermer la connexion
conn.close()
print("\n✓ Connexion à la base de données fermée")
print("\n" + "=" * 80)
print("✅ SCRIPT TERMINÉ AVEC SUCCÈS!")
print("=" * 80)


RÉSUMÉ ET STATISTIQUES FINALES

📊 RÉSUMÉ DES OPÉRATIONS:
--------------------------------------------------------------------------------
✓ Base de données créée: /content/drive/MyDrive/Colab Datasets/titanic_database.db
✓ Dataset téléchargé: titanic
✓ Table créée: passengers
✓ Total des lignes finales: 894
✓ Total des colonnes: 12

📈 OPÉRATIONS EFFECTUÉES:
--------------------------------------------------------------------------------
  • INSERT: 4 passagers insérés
  • SELECT: 5 requêtes de sélection exécutées
  • UPDATE: 3 requêtes de mise à jour exécutées
  • DELETE: 2 requêtes de suppression exécutées

📋 STATISTIQUES FINALES:
--------------------------------------------------------------------------------


,total,survivants,taux_survie,age_moyen,fare_moyen
0,717,292,40.73,29.73,35.54



✓ Connexion à la base de données fermée

✅ SCRIPT TERMINÉ AVEC SUCCÈS!


In [52]:
#Question 1
print("\n📋 TOP 5 PASSAGERS LES PLUS ÂGÉS AYANT SURVÉCU")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE Survived = 1
AND Age IS NOT NULL
ORDER BY Age DESC
LIMIT 5
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)



📋 TOP 5 PASSAGERS LES PLUS ÂGÉS AYANT SURVÉCU



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,631,1,1,"Barkworth, Mr. Algernon Henry Wilson",male,80.0,0,0,27042,33.00000,A23,S
1,276,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0,1,0,13502,85.75413,D7,S
2,484,1,3,"Turkula, Mrs. (Hedwig)",female,63.0,0,0,4134,9.58750,None,S
3,571,1,2,"Harris, Mr. George",male,62.0,0,0,S.W./PP 752,10.50000,None,S
4,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,88.00000,B28,None


In [53]:
#Question 2
print("\n📋 TARIF MOYEN PAR CLASSE")
print("=" * 80 + "\n")

query = f"""
SELECT Pclass,
       ROUND(AVG(Fare), 2) AS Tarif_Moyen
FROM {table_name}
GROUP BY Pclass
ORDER BY Pclass
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 TARIF MOYEN PAR CLASSE



,Pclass,Tarif_Moyen
0,1,95.26
1,2,20.61
2,3,13.66


In [54]:
#Question 3
print("\n📋 TAUX DE SURVIE PAR SEXE")
print("=" * 80 + "\n")

query = f"""
SELECT Sex,
       ROUND(AVG(Survived) * 100, 2) AS Taux_Survie_Pourcent
FROM {table_name}
GROUP BY Sex
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 TAUX DE SURVIE PAR SEXE



,Sex,Taux_Survie_Pourcent
0,female,74.29
1,male,18.97


In [55]:
#Question 4
print("\n📋 PASSAGERS DE PREMIÈRE CLASSE EMBARQUÉS À SOUTHAMPTON")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE Pclass = 1
AND Embarked = 'S'
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 PASSAGERS DE PREMIÈRE CLASSE EMBARQUÉS À SOUTHAMPTON



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,58.41000,C123,S
1,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,57.04875,E46,S
2,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,29.20500,C103,S
3,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,39.05000,A6,S
4,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,289.30000,C23 C25 C27,S
...,...,...,...,...,...,...,...,...,...,...,...,...
124,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,57.80962,D35,S
125,873,0,1,"Carlsson, Mr. Frans Olof",male,33.0,0,0,695,5.50000,B51 B53 B55,S
126,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,33.00000,B42,S
127,892,1,1,"Claude, Monsieur",male,40.0,1,1,AI001,605.00000,A123,S


In [56]:
#Question 5
print("\n📋 STATISTIQUES D'ÂGE PAR CLASSE")
print("=" * 80 + "\n")

query = f"""
SELECT Pclass,
       ROUND(MIN(Age),2) AS Age_Min,
       ROUND(MAX(Age),2) AS Age_Max,
       ROUND(AVG(Age),2) AS Age_Moyen
FROM {table_name}
GROUP BY Pclass
ORDER BY Pclass
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 STATISTIQUES D'ÂGE PAR CLASSE



,Pclass,Age_Min,Age_Max,Age_Moyen
0,1,0.92,80.0,37.06
1,2,0.67,70.0,29.96
2,3,0.42,74.0,25.14


In [57]:
#Question 6
print("\n📋 PASSAGERS AVEC FAMILLE À BORD")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE SibSp + Parch > 0
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 PASSAGERS AVEC FAMILLE À BORD



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25000,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,78.41163,C85,C
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,58.41000,C123,S
3,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.07500,None,S
4,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.13330,None,S
...,...,...,...,...,...,...,...,...,...,...,...,...
351,881,1,2,"Shelley, Mrs. William (Imanita Parrish Hall)",female,25.0,0,1,230433,26.00000,None,S
352,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.12500,None,Q
353,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45000,None,S
354,892,1,1,"Claude, Monsieur",male,40.0,1,1,AI001,605.00000,A123,S


In [58]:
#Question 7
print("\n📋 ENFANTS DE MOINS DE 10 ANS")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE Age < 10
ORDER BY Age
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 ENFANTS DE MOINS DE 10 ANS



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,804,1,3,"Thomas, Master. Assad Alexander",male,0.42,0,1,2625,8.5167,None,C
1,756,1,2,"Hamalainen, Master. Viljo",male,0.67,1,1,250649,14.5000,None,S
2,470,1,3,"Baclini, Miss. Helene Barbara",female,0.75,2,1,2666,19.2583,None,C
3,645,1,3,"Baclini, Miss. Eugenie",female,0.75,2,1,2666,19.2583,None,C
4,79,1,2,"Caldwell, Master. Alden Gates",male,0.83,0,2,248738,29.0000,None,S
...,...,...,...,...,...,...,...,...,...,...,...,...
57,481,0,3,"Goodwin, Master. Harold Victor",male,9.00,5,2,CA 2144,46.9000,None,S
58,490,1,3,"Coutts, Master. Eden Leslie ""Neville""",male,9.00,1,1,C.A. 37671,15.9000,None,S
59,542,0,3,"Andersson, Miss. Ingeborg Constanzia",female,9.00,4,2,347082,31.2750,None,S
60,635,0,3,"Skoog, Miss. Mabel",female,9.00,3,2,347088,27.9000,None,S


In [59]:
#Question 8
print("\n📋 TAUX DE SURVIE PAR SEXE ET CLASSE")
print("=" * 80 + "\n")

query = f"""
SELECT Sex,
       Pclass,
       ROUND(AVG(Survived) * 100, 2) AS Taux_Survie_Pourcent
FROM {table_name}
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 TAUX DE SURVIE PAR SEXE ET CLASSE



,Sex,Pclass,Taux_Survie_Pourcent
0,female,1,96.84
1,female,2,92.11
2,female,3,50.00
3,male,1,37.40
4,male,2,15.60
5,male,3,13.51


In [60]:
#Question 9
print("\n📋 PASSAGERS SANS NUMÉRO DE CABINE")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE Cabin IS NULL
OR Cabin = ''
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)


📋 PASSAGERS SANS NUMÉRO DE CABINE



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
2,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S
3,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,None,Q
4,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,None,S
...,...,...,...,...,...,...,...,...,...,...,...,...
684,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,None,S
685,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,None,S
686,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,None,Q
687,893,0,3,"Dupont, Monsieur",male,25.0,0,0,AI002,7.7500,None,C


In [61]:
#Question 10
print("\n📋 TOP 10 DES TARIFS LES PLUS ÉLEVÉS")
print("=" * 80 + "\n")

query = f"""
SELECT *
FROM {table_name}
WHERE Fare IS NOT NULL
ORDER BY Fare DESC
LIMIT 10
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)



📋 TOP 10 DES TARIFS LES PLUS ÉLEVÉS



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,1,1,"Claude, Monsieur",male,40.0,1,1,AI001,605.00000,A123,S
1,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,563.56212,None,C
2,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,563.56212,B51 B53 B55,C
3,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,563.56212,B101,C
4,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,289.30000,C23 C25 C27,S
5,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,289.30000,C23 C25 C27,S
6,342,1,1,"Fortune, Miss. Alice Elizabeth",female,24.0,3,2,19950,289.30000,C23 C25 C27,S
7,439,0,1,"Fortune, Mr. Mark",male,64.0,1,4,19950,289.30000,C23 C25 C27,S
8,312,1,1,"Ryerson, Miss. Emily Borie",female,18.0,2,2,PC 17608,288.61250,B57 B59 B63 B66,C
9,743,1,1,"Ryerson, Miss. Susan Parker ""Suzette""",female,21.0,2,2,PC 17608,288.61250,B57 B59 B63 B66,C


---

## 📚 Ressources Supplémentaires

### Fichiers Générés
Vos données sont sauvegardées ici :
```
Google Drive > Colab Datasets > titanic_database.db
```

### Documentation
- **README_COLAB.md** : Guide complet
- **sql_queries_examples.py** : 30+ requêtes SQL
- **20_Exercices_SQL.docx** : Exercices pratiques

### Prochaines Étapes
1. Modifiez les données selon vos besoins
2. Testez d'autres datasets (Iris, Housing, Adult)
3. Explorez les requêtes SQL avancées
4. Créez vos propres analyses

### Datasets Recommandés
- Titanic (893 lignes) : Débutant
- Iris (150 lignes) : Classification simple
- Housing (20K lignes) : Régression
- Adult (32K lignes) : Données complexes

---

**Bon learning! 🚀**